# TP 1 — Du texte brut au premier modèle bigramme

**Objectif** : comprendre comment un corpus devient un ensemble de prédictions supervisées, puis entraîner une table qui prédit le caractère suivant.

**Livrable** : un notebook exécutable de haut en bas, avec deux générations (avant et après entraînement) et des réponses courtes aux questions d'interprétation.

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

## Étape 1 — Télécharger et inspecter le corpus

À relever : la longueur du corpus, les retours à la ligne, la ponctuation, ainsi que les majuscules et minuscules.

In [2]:
from urllib.request import urlretrieve
import os

url = ("https://raw.githubusercontent.com/karpathy/"
       "char-rnn/master/data/tinyshakespeare/input.txt")
urlretrieve(url, "input.txt")
print(os.path.abspath("input.txt"))

with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(len(text))
print(text[:1000])

h:\Documents\CY Tech\ING 3\NLP\TPs\input.txt
1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak t

> **Questions d'interprétation**
> 1. Quelle est ici l'unité d'un token ?
> 2. Pourquoi préciser `encoding="utf-8"` ?
> 3. Le fichier est-il déjà séparé en exemples ?

**Réponses**
1. **Unité d'un token** : un **caractère** (lettre, chiffre, ponctuation, espace, `\n`). Le vocabulaire contient donc 65 tokens.
2. **`encoding="utf-8"`** : un fichier n'est qu'une suite d'octets, il faut préciser comment les transformer en caractères. Sans cela, Python utilise l'encodage par défaut du système, qui varie d'une machine à l'autre et peut mal lire (ou refuser) certains caractères.
3. **Déjà séparé en exemples ?** Non : c'est un seul long texte d'environ 1,1 M de caractères. Les exemples (contexte → cible) sont fabriqués par nous, en découpant des fenêtres dans ce texte (étapes 3 et 4).

## Étape 2 — Passer des caractères aux entiers

In [3]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join([ itos[i] for i in ids])

message = "hii there"
print(encode(message))
print(decode(encode(message)))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


> **Questions d'interprétation**
> 1. Pourquoi trier `chars` ?
> 2. Que garantit le test `decode(encode(message)) == message` ?

**Réponses**
1. **Pourquoi trier ?** `set(text)` n'a pas d'ordre garanti : sans tri, un même caractère pourrait recevoir un identifiant différent à chaque exécution. Le tri rend la correspondance caractère ↔ entier **déterministe et reproductible**, ce qui est indispensable pour réutiliser un modèle entraîné.
2. **Ce que garantit le test** : le codage est **sans perte**. `stoi` et `itos` sont exactement inverses, chaque caractère a un identifiant unique et on retrouve le texte d'origine. Ceci n'est valable que pour les caractères présents dans le corpus : un caractère inconnu provoquerait une `KeyError`.

## Étape 3 — Fabriquer les prédictions attendues

Le jeu de données est séparé en deux parties : **90 %** pour l'entraînement (`train`) et **10 %** pour la validation (`val`).

In [4]:
t = [1,2,3,4,5,6,7,8,9]
block_size = 8
x = t[:block_size]
y = t[1:block_size + 1]
print(x)
print(y)

[1, 2, 3, 4, 5, 6, 7, 8]
[2, 3, 4, 5, 6, 7, 8, 9]


In [5]:
data = torch.tensor(encode(text), dtype=torch.long)

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size + 1]

for t in range(block_size):
    context = x[:t + 1]
    target = y[t]
    print(context, "->", target)
    

tensor([18]) -> tensor(47)
tensor([18, 47]) -> tensor(56)
tensor([18, 47, 56]) -> tensor(57)
tensor([18, 47, 56, 57]) -> tensor(58)
tensor([18, 47, 56, 57, 58]) -> tensor(1)
tensor([18, 47, 56, 57, 58,  1]) -> tensor(15)
tensor([18, 47, 56, 57, 58,  1, 15]) -> tensor(47)
tensor([18, 47, 56, 57, 58,  1, 15, 47]) -> tensor(58)


> **Questions d'interprétation**
> 1. Combien de prédictions produit ce bloc ?
> 2. Pourquoi ne pas mélanger train et validation ?
> 3. Pour t = 3, quels sont le contexte et la cible ?

**Réponses**
1. **`block_size` = 8 prédictions** : un bloc de 9 tokens donne 8 couples (contexte → cible), avec des contextes de longueur 1 à 8.
2. **Pas de mélange train/validation** : la validation doit contenir du texte **jamais vu** pendant l'entraînement. Sinon, une loss basse pourrait venir de la mémorisation, et on ne mesurerait plus la capacité à **généraliser** (le surapprentissage ne serait pas détecté).
3. **t = 3** : contexte = `[18, 47, 56, 57]` (« Firs »), cible = `58` (« t »).

## Étape 4 — Échantillonner un mini-batch

In [6]:
batch_size = 4
block_size = 8

def get_batch(split):
    source = train_data if split == "train" else val_data
    starts = torch.randint(
        len(source) - block_size, (batch_size,)
    )
    x = torch.stack([
        source[i : i + block_size] for i in starts
    ])
    y = torch.stack([
        source[i + 1 : i + block_size + 1] for i in starts
    ])
    return x, y

xb, yb = get_batch("train")
print(xb.shape, yb.shape)

torch.Size([4, 8]) torch.Size([4, 8])


> **Questions d'interprétation**
> 1. Pourquoi retirer `block_size` à la borne de tirage ?
> 2. Que changerait B = 32 ?
> 3. Que changerait T = 64 ?

**Réponses**
1. **Borne de tirage** : chaque départ `i` doit laisser la place à `block_size` tokens pour `x` **et** à un token de plus pour `y` (décalé d'une position). L'indice le plus éloigné lu est `i + block_size`, qui doit exister : `i ≤ len(source) − block_size − 1`. Comme `randint` exclut sa borne haute, `len(source) − block_size` donne exactement le départ maximal valide. Sans cela, les tranches en fin de texte seraient plus courtes et `torch.stack` échouerait.
2. **B = 32** : 32 séquences indépendantes par batch au lieu de 4, soit 32 × 8 = 256 prédictions par pas. Le gradient est moins bruité (moyenne sur plus d'exemples) et chaque pas coûte plus de calcul et de mémoire. Le contexte de chaque prédiction ne change pas.
3. **T = 64** : des séquences de 64 caractères, donc plus de prédictions par séquence et des contextes jusqu'à 64 caractères. Pour le bigramme, cela ne change rien à ce qu'il peut apprendre (il ne regarde que le dernier caractère). Pour le Transformer du TP 4, cela allonge la mémoire disponible, mais aussi le coût de l'attention (en T²).

## Étape 5 — Une table produit les logits

In [7]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(
            vocab_size, vocab_size
        )

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)
        # idx: (B,T) -> logits: (B, T, C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

> **Questions d'interprétation**
> 1. Pourquoi une table C × C ici ?
> 2. Où retrouvera-t-on `d_emb` et la couche `lm_head` dans le TP 4 ?

**Réponses**
1. **C × C** : une ligne par caractère **courant** (l'entrée choisit la ligne) et une colonne par caractère **suivant possible** (les scores, ou logits). Entrée et sortie appartiennent au même vocabulaire, donc la table est `vocab_size × vocab_size` (65 × 65). La case (a, b) est le score de « b vient après a ».
2. **Dans le TP 4**, cette table est séparée en deux :
   - `token_embedding_table = nn.Embedding(vocab_size, n_embd)` : chaque caractère devient un vecteur de dimension **d_emb** (64) ;
   - `lm_head = nn.Linear(n_embd, vocab_size)` : ramène ce vecteur vers les 65 logits.

   Entre les deux s'insèrent les blocs (attention + MLP).

## Étapes 6 et 7 — Générer, puis apprendre

Procédure : générer une première fois avec les poids aléatoires, entraîner la table, puis relancer exactement la même génération pour observer ce qui a changé.

### Étape 6 — Génération (méthode de `BigramLanguageModel`)

In [8]:
def generate(self, idx, max_new_tokens):
    for _ in range(max_new_tokens):
        logits, _ = self(idx)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(
            probs, num_samples=1
        )
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

BigramLanguageModel.generate = generate

In [9]:
# Génération avant entraînement (poids aléatoires)
model = BigramLanguageModel(vocab_size)
idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(model.generate(idx, max_new_tokens=100)[0].tolist()))


e!ocWZ&K G;CSGyv.uycu''KQ
rOi$bXUSqQ WLVJs3lmNOKj:mDpCJs;hCSGG!:rfgDbFHTsZmz.j:BCifrdp,PZp
pSXPaS.?O


### Étape 7 — Entraînement

In [10]:
optimizer = torch.optim.AdamW(
    model.parameters(), lr=1e-3
)

for step in range(1000):
    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

4.131296634674072


In [11]:
# Génération après entraînement
print(decode(model.generate(idx, max_new_tokens=100)[0].tolist()))


 D:DJwnVACdA:XlD:wLljPCWid!QH WL'aiokAlV;DLYpleDepmeOEJeEQyopjF?NIbnD'enBX&YXUMZ'Y-'&In'-WrRtrldAY-s


> **Bilan du TP 1**
> 1. Pourquoi ne garde-t-on que le dernier pas de temps pendant la génération ?
> 2. Pourquoi aplatit-on B × T pendant le calcul de la loss ?
> 3. Comparez la loss et la structure du texte avant/après 100 étapes, puis avec 1 000 étapes.

**Réponses**
1. **Dernier pas de temps** : `logits` contient une prédiction pour chaque position, mais les positions `t < T−1` prédisent des tokens **déjà connus**. Seule la dernière position prédit le token **inconnu** à générer. `logits[:, -1, :]` fait passer de `(B, T, C)` à `(B, C)`, la forme attendue par `softmax` puis `multinomial`.
2. **Aplatir B × T** : `F.cross_entropy` attend des logits `(N, C)` et des cibles `(N,)`. Chaque position de chaque séquence est une prédiction indépendante : on les met toutes « à la suite » (N = B·T) pour calculer la loss moyenne en une fois.
3. **Avant / après entraînement** :
   - Une prédiction uniforme donnerait une loss de `ln(65) ≈ 4,17`. Au départ, la loss est en général un peu **au-dessus** (≈ 4,5–4,9), car la table est initialisée avec des valeurs aléatoires N(0, 1) : le modèle n'est pas uniforme mais « confiant au hasard ». Le texte généré est alors une suite aléatoire de caractères.
   - Après 1 000 étapes (`lr = 1e-3`, `batch_size = 4`), la loss affichée est **3,86** : elle passe sous ln(65), donc le modèle fait déjà mieux qu'un tirage uniforme. La baisse reste modeste : chaque étape ne voit que 4 × 8 = 32 prédictions, et 1 000 petites mises à jour ne suffisent pas à bien remplir une table de 65 × 65 = 4 225 scores. Le texte reste donc très proche de l'aléatoire, avec un peu plus d'espaces et de retours à la ligne. Il faut beaucoup plus d'étapes (≈ 10 000) ou un batch plus grand pour voir apparaître des paires de lettres plausibles.
   - Avec seulement 100 étapes, la loss bouge à peine par rapport au départ.

   *Remarque* : `loss.item()` est la loss d'**un seul** mini-batch, c'est donc une valeur bruitée.